In [ ]:
import sys

print(sys.executable)

In [8]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings

## Indexing

In [9]:
# WebBAseLoader -> downloads given URL and parses the HTML
# load documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()
docs

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/user/Learnings/Repo/Learning/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3579, in run_code
  File "/tmp/ipykernel_139023/585498121.py", line 11, in <module>
    docs = loader.load()
  File "/home/user/Learnings/Repo/Learning/.venv/lib/python3.10/site-packages/langchain_core/document_loaders/base.py", line 43, in load
  File "/home/user/Learnings/Repo/Learning/.venv/lib/python3.10/site-packages/langchain_community/document_loaders/web_base.py", line 375, in lazy_load
  File "/home/user/Learnings/Repo/Learning/.venv/lib/python3.10/site-packages/langchain_community/document_loaders/web_base.py", line 357, in _scrape
  File "/home/user/Learnings/Repo/Learning/.venv/lib/python3.10/site-packages/requests/sessions.py", line 602, in get
  File "/home/user/Learnings/Repo/Learning/.venv/lib/python3.10/site-packages/requests/sessions.py", line 589, in request
  File "/home/user/Learnings/Repo/Learning/.venv/lib/python

In [4]:
# Split the document into chunks
# chunk_size=1000: target max size of each chunk (in characters).
# chunk_overlap=200: each chunk overlaps the previous by 200 characters to preserve context at boundaries.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
splits

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refi

In [5]:
# printing each chunk size
for idx, i in enumerate(splits):
    print(idx, len(i.page_content))

0 969
1 665
2 939
3 987
4 760
5 974
6 858
7 960
8 412
9 989
10 735
11 545
12 962
13 980
14 452
15 542
16 760
17 772
18 818
19 677
20 916
21 814
22 380
23 855
24 805
25 639
26 456
27 610
28 608
29 679
30 726
31 971
32 195
33 997
34 828
35 697
36 947
37 541
38 961
39 704
40 556
41 958
42 666
43 664
44 983
45 127
46 936
47 999
48 310
49 18
50 48
51 991
52 996
53 476
54 702
55 989
56 378
57 132
58 808
59 568
60 976
61 956
62 940


In [30]:
# Embed
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") not using because of ratelimit
# using huggingface
vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}